# Yolo - PreTrained

In [3]:
from ultralytics import YOLO
import os
import json
import shutil

In [3]:
!pip install ultralytics

## Criar yaml para teste com o respetivo formato para Ultralytics YOLO 

Fazer yaml para os dados 
https://docs.ultralytics.com/pt/datasets/detect/#ultralytics-yolo-format

In [2]:
json_path = "annotations.json"
output_root = ""

# Criação de pastas
splits = ['train', 'val', 'test']
for split in splits:
    os.makedirs(os.path.join(output_root, "images", split), exist_ok=True)
    os.makedirs(os.path.join(output_root, "labels", split), exist_ok=True)

# Carregar ficheiro JSON
with open(json_path, 'r') as f:
    data = json.load(f)

# Criar índice por id para acesso rápido
id_to_image = {img['id']: img for img in data['images']}

annotations = data['annotations']['pieces']

# Criação de índice de anotações por imagem_id
annotations_by_image = {}
for ann in annotations:
    image_id = ann['image_id']
    if ann.get('bbox'):
        bbox = ann['bbox']
    else:
        continue
    annotations_by_image.setdefault(image_id, []).append(bbox)

# Processar splits
for split in splits:
    ids = data['splits']['chessred2k'].get(split, [])
    for image_id in ids['image_ids']:
        image_info = id_to_image.get(image_id)
        if not image_info:
            print(f"[AVISO] ID {image_id} não encontrado em 'images'")
            continue

        src_path = "chessred2k/" + image_info.get('path')
        if not src_path or not os.path.isfile(src_path):
            print(f"[AVISO] Ficheiro não encontrado: {src_path}")
            continue

        dst_path = os.path.join(output_root, "Images", split, image_info['file_name'])
        shutil.copy2(src_path, dst_path)

        print(f"[OK] Copiada: {src_path} -> {dst_path}")

        img_width = image_info['width']
        img_height = image_info['height']

        label_path = os.path.join(output_root, "Labels", split, image_info['file_name'].replace(".jpg", ".txt"))

        with open(label_path, 'w') as label_file:
            for bbox in annotations_by_image.get(image_id, []):
                x, y, w, h = bbox

                # Converter para formato YOLO
                x_center = (x + w / 2) / img_width
                y_center = (y + h / 2) / img_height
                w_norm = w / img_width
                h_norm = h / img_height

                label_file.write(f"0 {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")

[OK] Copiada: chessred2k/images/19/G019_IMG000.jpg -> Images\train\G019_IMG000.jpg
[OK] Copiada: chessred2k/images/19/G019_IMG001.jpg -> Images\train\G019_IMG001.jpg
[OK] Copiada: chessred2k/images/19/G019_IMG002.jpg -> Images\train\G019_IMG002.jpg
[OK] Copiada: chessred2k/images/19/G019_IMG003.jpg -> Images\train\G019_IMG003.jpg
[OK] Copiada: chessred2k/images/19/G019_IMG004.jpg -> Images\train\G019_IMG004.jpg
[OK] Copiada: chessred2k/images/19/G019_IMG005.jpg -> Images\train\G019_IMG005.jpg
[OK] Copiada: chessred2k/images/19/G019_IMG006.jpg -> Images\train\G019_IMG006.jpg
[OK] Copiada: chessred2k/images/19/G019_IMG007.jpg -> Images\train\G019_IMG007.jpg
[OK] Copiada: chessred2k/images/19/G019_IMG008.jpg -> Images\train\G019_IMG008.jpg
[OK] Copiada: chessred2k/images/19/G019_IMG009.jpg -> Images\train\G019_IMG009.jpg
[OK] Copiada: chessred2k/images/19/G019_IMG010.jpg -> Images\train\G019_IMG010.jpg
[OK] Copiada: chessred2k/images/19/G019_IMG011.jpg -> Images\train\G019_IMG011.jpg
[OK]

## Train with a pre-trained model

In [5]:
model = YOLO("yolo11n.yaml")  # build a new model from YAML
model = YOLO("yolo11n.pt")  # load a pretrained model
model = YOLO("yolo11n.yaml").load("yolo11n.pt")  # build from YAML and transfer weights

100%|██████████| 5.35M/5.35M [00:00<00:00, 6.60MB/s]


Transferred 499/499 items from pretrained weights


In [17]:
model.train(data="data.yaml", epochs=100, imgsz=640)

Ultralytics 8.3.153  Python-3.12.9 torch-2.7.1+cpu CPU (Intel Core(TM) i7-14700KF)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.yaml, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrained=yolo11n.pt, prof

train: Scanning C:\Users\diogo\mestrado\2º semestre\VC\Project\Part 2 e 3\Task 3\1st_part\labels\train... 1442 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1442/1442 [00:00<00:00, 2889.55it/s]

train: New cache created: C:\Users\diogo\mestrado\2 semestre\VC\Project\Part 2 e 3\Task 3\1st_part\labels\train.cache
val: Fast image access  (ping: 0.00.0 ms, read: 3392.4234.9 MB/s, size: 1766.4 KB)



c:\Users\diogo\anaconda3\envs\vc\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
val: Scanning C:\Users\diogo\mestrado\2º semestre\VC\Project\Part 2 e 3\Task 3\1st_part\labels\val... 330 images, 0 backgrounds, 0 corrupt: 100%|██████████| 330/330 [00:00<00:00, 3188.36it/s]


val: New cache created: C:\Users\diogo\mestrado\2 semestre\VC\Project\Part 2 e 3\Task 3\1st_part\labels\val.cache
Plotting labels to runs\detect\train\labels.jpg... 


...

                   all        330       6132          1          1      0.995      0.851

100 epochs completed in 5.768 hours.
Optimizer stripped from runs\detect\train\weights\last.pt, 5.5MB


Optimizer stripped from runs\detect\train\weights\best.pt, 5.5MB

Validating runs\detect\train\weights\best.pt...
Ultralytics 8.3.153  Python-3.12.9 torch-2.7.1+cpu CPU (Intel Core(TM) i7-14700KF)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 11/11 [00:18<00:00,  1.70s/it]


                   all        330       6132      0.999          1      0.995      0.875
Speed: 0.5ms preprocess, 24.4ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to runs\detect\train


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000018E545F9880>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [13]:
# Load a model
model = YOLO("runs/detect/train/weights/best.pt")  # load a custom model

# Validate the model
metrics = model.val()  # no arguments needed, dataset and settings remembered
metrics.box.map  # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps

# Teste the model
metrics = model.val(split="test")  
metrics.box.map  # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps

Ultralytics 8.3.153  Python-3.13.3 torch-2.7.1+cpu CPU (Intel Core(TM) Ultra 7 165U)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.40.1 ms, read: 1004.8190.4 MB/s, size: 2187.5 KB)


val: Scanning C:\Users\Dell\OneDrive - Confidencial Imobiliario\Documentos\ComputerVision\Task 3\1st_part\labels\val.cache... 330 images, 0 backgrounds, 0 corrupt: 100%|██████████| 330/330 [00:00<?, ?it/s]
c:\Users\Dell\OneDrive - Confidencial Imobiliario\Documentos\ComputerVision\Task 3\1st_part\.venv\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 21/21 [00:42<00:00,  2.04s/it]


                   all        330       6132      0.999          1      0.995      0.875
Speed: 0.7ms preprocess, 55.8ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to runs\detect\val9
Ultralytics 8.3.153  Python-3.13.3 torch-2.7.1+cpu CPU (Intel Core(TM) Ultra 7 165U)
val: Fast image access  (ping: 0.20.1 ms, read: 1854.191.6 MB/s, size: 2251.3 KB)


val: Scanning C:\Users\Dell\OneDrive - Confidencial Imobiliario\Documentos\ComputerVision\Task 3\1st_part\labels\test.cache... 306 images, 0 backgrounds, 0 corrupt: 100%|██████████| 306/306 [00:00<?, ?it/s]
c:\Users\Dell\OneDrive - Confidencial Imobiliario\Documentos\ComputerVision\Task 3\1st_part\.venv\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 20/20 [01:09<00:00,  3.47s/it]


                   all        306       6294      0.999      0.999      0.995      0.862
Speed: 1.0ms preprocess, 102.5ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to runs\detect\val10


array([    0.86151])

In [14]:
metrics

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000020CB1F70640>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

Colocar imagens dos resultados e uns graficos de metricas

## Train from scrath

In [3]:
model2 = YOLO("yolo11n.yaml")  # build a new model from YAML

In [ ]:
model2.train(data="data.yaml", epochs=100, imgsz=640)

Ultralytics 8.3.153  Python-3.12.9 torch-2.7.1+cpu CPU (Intel Core(TM) i7-8750H 2.20GHz)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.yaml, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrained=True, prof

100%|██████████| 755k/755k [00:00<00:00, 4.88MB/s]


Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]           
  7                  -1  1    295424  ultralytics

train: Scanning C:\Users\Hp\mestrado\VC\Task 3\1st_part\labels\train... 1442 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1442/1442 [00:01<00:00, 942.94it/s] 


train: New cache created: C:\Users\Hp\mestrado\VC\Task 3\1st_part\labels\train.cache
val: Fast image access  (ping: 0.10.0 ms, read: 611.799.6 MB/s, size: 1766.4 KB)


...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     69/100         0G     0.4423     0.2683     0.8159         37        640: 100%|██████████| 91/91 [13:11<00:00,  8.70s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 11/11 [01:14<00:00,  6.81s/it]

                   all        330       6132      0.999          1      0.995      0.851



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     70/100         0G     0.4469     0.2694     0.8159        591        640:  56%|█████▌    | 51/91 [07:31<06:08,  9.21s/it]

In [ ]:
model2 = YOLO("runs/detect/train2/weights/last.pt")
model2.train(resume=True)

Ultralytics 8.3.153  Python-3.12.9 torch-2.7.1+cpu CPU (Intel Core(TM) i7-8750H 2.20GHz)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs\detect\train\weights\last.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, 

train: Scanning C:\Users\Hp\mestrado\VC\Task 3\1st_part\labels\train.cache... 1442 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1442/1442 [00:00<?, ?it/s]

val: Fast image access  (ping: 0.10.0 ms, read: 744.4119.8 MB/s, size: 1766.4 KB)



c:\Users\Hp\anaconda3\envs\vc\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
val: Scanning C:\Users\Hp\mestrado\VC\Task 3\1st_part\labels\val.cache... 330 images, 0 backgrounds, 0 corrupt: 100%|██████████| 330/330 [00:00<?, ?it/s]
c:\Users\Hp\anaconda3\envs\vc\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Plotting labels to runs\detect\train\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
Resuming training runs\detect\train\weights\last.pt from epoch 70 to 100 total epochs
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to runs\detect\train
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

                   all        330       6132          1          1      0.995      0.849



31 epochs completed in 7.255 hours.
Optimizer stripped from runs\detect\train\weights\last.pt, 5.5MB
Optimizer stripped from runs\detect\train\weights\best.pt, 5.5MB

Validating runs\detect\train\weights\best.pt...
Ultralytics 8.3.153  Python-3.12.9 torch-2.7.1+cpu CPU (Intel Core(TM) i7-8750H 2.20GHz)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 11/11 [01:01<00:00,  5.62s/it]


                   all        330       6132      0.998      0.998      0.995       0.87
Speed: 4.7ms preprocess, 116.8ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to runs\detect\train


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000016BD4C19910>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [ ]:
# Load a model
model = YOLO("runs/detect/train2/weights/best.pt")  # load a custom model

# Validate the model
metrics = model.val()  # no arguments needed, dataset and settings remembered
metrics.box.map  # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps

# Teste the model
metrics = model.val(split="test")  
metrics.box.map  # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps

Ultralytics 8.3.153  Python-3.13.3 torch-2.7.1+cpu CPU (Intel Core(TM) Ultra 7 165U)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.30.1 ms, read: 169.977.4 MB/s, size: 2291.1 KB)


val: Scanning C:\Users\Dell\OneDrive - Confidencial Imobiliario\Documentos\ComputerVision\Task 3\1st_part\labels\val.cache... 330 images, 0 backgrounds, 0 corrupt: 100%|██████████| 330/330 [00:00<?, ?it/s]
c:\Users\Dell\OneDrive - Confidencial Imobiliario\Documentos\ComputerVision\Task 3\1st_part\.venv\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 21/21 [00:50<00:00,  2.41s/it]


                   all        330       6132      0.998      0.998      0.995       0.87
Speed: 0.7ms preprocess, 57.3ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to runs\detect\val7
Ultralytics 8.3.153  Python-3.13.3 torch-2.7.1+cpu CPU (Intel Core(TM) Ultra 7 165U)
val: Fast image access  (ping: 0.30.0 ms, read: 183.735.5 MB/s, size: 2182.9 KB)


val: Scanning C:\Users\Dell\OneDrive - Confidencial Imobiliario\Documentos\ComputerVision\Task 3\1st_part\labels\test.cache... 306 images, 0 backgrounds, 0 corrupt: 100%|██████████| 306/306 [00:00<?, ?it/s]
c:\Users\Dell\OneDrive - Confidencial Imobiliario\Documentos\ComputerVision\Task 3\1st_part\.venv\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 20/20 [00:51<00:00,  2.57s/it]


                   all        306       6294      0.998      0.996      0.995      0.858
Speed: 0.6ms preprocess, 64.4ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to runs\detect\val8


array([    0.85829])

In [11]:
metrics

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000020CB7216210>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480